# **6. Wiring the forecast into the app**

Chapters 2 to 5 produced a validated forecaster and an advisor. Chapter 6 makes them part of
Loca-T: a service the other layers can call, an endpoint for the dashboard, a USSD menu and
an SMS digest for hosts. The app code lives in `src/locat/`; this notebook records each step
and checks it by calling the running server.

## **6.1 The advice service and its endpoint**

### **6.1.1 Why**
The forecaster and advisor are modules, but nothing in the app calls them. The first piece
is a service that fits the forecaster once, forecasts for a property starting on demo day,
and runs the advisor. A JSON endpoint exposes it, so the dashboard and later steps use one
path.

### **6.1.2 What gets built**
1. `advice_service.py`: loads the saved data and fits the forecaster once (cached). Forecasts
   start today, or on 26 November if today is earlier (the day after the demo history ends),
   and never beyond 60 days or the end of the calendar (31 March 2027).
2. `GET /advice/{listing_code}?days=N`: night-by-night forecast, range, level, action, price and
   reasons, labelled as synthetic data with the validated horizon.
3. A setting `DEMO_TODAY` to fix the demo date for rehearsals (empty means today).
4. Three tests on the service.

### **6.1.3 Predictions (written before the run)**
1. `pytest` reports 8 passed (5 for the advisor, 3 new).
2. `GET /advice/XESD2C?days=28` returns 200 and 28 nights, starting on 2026-11-26 (while today
   is on or before that date), with `synthetic_data` true.
3. With `days=60`: 60 nights, the last on 2027-01-24, and actions of hold 41, promote 15 and
   test 4, with 14 busy nights, exactly as in 5.11.
4. The festive rows (25 and 26 December, 1 and 2 January) are test at KES 4,700, with ranges
   of 85-98% and 83-98%.
5. An unknown listing code returns 404, and `days=500` returns 60 nights, not 500.

### **6.1.4 Decision rule**
If all five hold, the service is accepted and step 6.3 adds the USSD outlook and the SMS
digest. If a prediction fails, we find out why first.

In [1]:
import httpx
import pandas as pd

BASE = "http://127.0.0.1:8020"
CODE = "XESD2C"  # Bibi Akinyi Homestay's listing code from Phase 4a

r28 = httpx.get(f"{BASE}/advice/{CODE}", params={"days": 28}, timeout=120)
r60 = httpx.get(f"{BASE}/advice/{CODE}", params={"days": 60}, timeout=120)
print("status:", r28.status_code, r60.status_code)
d60 = r60.json()
print({k: d60[k] for k in ["listing_code", "property", "destination", "base_price",
                           "synthetic_data", "validated_horizon_days"]})

n28, n60 = pd.DataFrame(r28.json()["nights"]), pd.DataFrame(d60["nights"])
print(f"\nnights: {len(n28)} and {len(n60)} | first night {n60['date'].iloc[0]} | last night {n60['date'].iloc[-1]}")
print("actions (60 days):", n60["action"].value_counts().to_dict(),
      "| busy nights:", int((n60["level"] == "busy").sum()))

festive = n60[n60["date"].between("2026-12-24", "2027-01-03")]
print("\nFestive nights:")
print(festive[["day", "forecast_pct", "lower_pct", "upper_pct", "level",
               "action", "suggested_price", "reasons"]].to_string(index=False))

print("\nunknown code:", httpx.get(f"{BASE}/advice/ZZZZZZ").status_code)
big = httpx.get(f"{BASE}/advice/{CODE}", params={"days": 500}, timeout=120).json()
print("days=500 returns", len(big["nights"]), "nights")

status: 200 200
{'listing_code': 'XESD2C', 'property': 'Bibi Akinyi Homestay', 'destination': 'Naivasha', 'base_price': 4500.0, 'synthetic_data': True, 'validated_horizon_days': 60}

nights: 28 and 60 | first night 2026-11-26 | last night 2027-01-24
actions (60 days): {'hold': 41, 'promote': 15, 'test': 4} | busy nights: 14

Festive nights:
       day  forecast_pct  lower_pct  upper_pct  level action  suggested_price                                             reasons
Thu 24 Dec            44         38         51 normal   hold             4500                                     school holidays
Fri 25 Dec            97         85         98   busy   test             4700 school holidays, long weekend, weekend, payday week
Sat 26 Dec            96         85         98   busy   test             4700 school holidays, long weekend, weekend, payday week
Sun 27 Dec            63         55         74   busy   hold             4500          school holidays, long weekend, payday week
Mon 28 

## **6.2 Review of results: the advice service and endpoint (step 6.1)**

### **6.2.1 Verdict**
All five predictions held. The service gives the app exactly the numbers the notebooks
produced, the endpoint behaves correctly at its edges, and the work is committed. Two gaps
remain, and the next step closes one of them: the first request pays for fitting the
forecaster, and the endpoint is open to anyone who knows a listing code.

### **6.2.2 Prediction scorecard**

| Prediction (6.1.3) | Result | Verdict |
|---|---|---|
| `pytest` reports 8 passed | 8 passed in 3.03 seconds | Held |
| `days=28` returns 200 and 28 nights from 2026-11-26, synthetic flag true | Same; the 7-day call also worked | Held |
| `days=60`: 60 nights to 2027-01-24; hold 41, promote 15, test 4; 14 busy | Exactly those | Held |
| Festive rows are tests at KES 4,700 with ranges 85-98% and 83-98% | Exactly those, with the same reasons | Held |
| Unknown code gives 404; `days=500` gives 60 nights | 404; 60 nights | Held |

### **6.2.3 What the numbers show**

#### **6.2.3.1 The app and the notebook agree**
The endpoint's 60-night result reproduces the notebook result from step 5.11 night for night:
the same counts, the same festive forecasts and ranges, the same price of KES 4,700. That
is the integration test that matters: no number changed on the way into the app.

#### **6.2.3.2 The edges**
`days=500` is clipped to the validated 60, an unknown code is a clean 404, and the calendar
limit is enforced by the service (the tests cover a start date beyond 31 March 2027).

#### **6.2.3.3 The demo clock**
Today is 16 November, yet the first night returned is 26 November. That is by design: the
demo history ends on 25 November, so the forecast starts on the next day. From 26 November
the service starts at today, and `DEMO_TODAY` can fix the date for rehearsals.

### **6.2.4 Gaps**
- **First-request latency.** The forecaster is fitted when first needed. The tests suggest it
  is a few seconds, which is too slow for the first USSD request in a demo. Step 6.3 fits it at
  server start.
- **No authentication.** Anyone with a listing code can read the outlook. The data are
  synthetic, so this is acceptable for now, but the hardening step must tie it to the host.
  The endpoint that sends SMS (next) is protected from the start, because it costs money
  and can be used to spam a host.

### **6.2.5 Decision**
Accept the service and the endpoint. We are on the plan from 12 November: today and
tomorrow cover the USSD outlook and the SMS digest, then the dashboard.

## **6.3 Next step: the outlook on a phone**

### **6.3.1 Why**
Hosts like Bibi Akinyi use a basic phone. The outlook is only worth building if she can ask
for it on USSD and receive it by SMS, in her own language.

### **6.3.2 What gets built**
1. **USSD option 5, "My outlook"** (hosts only): one screen with the busiest nights in the next
   two weeks, the peak night in the next 60 days and, where the advisor has one, the price to try.
2. **An SMS digest** in two parts of at most 306 characters: the same facts plus the quiet
   stretch (the dates of the promote nights), ending with a note that the data are demo data.
3. **`GET /advice/{code}/digest`**: previews the text without sending anything.
4. **`POST /advice/{code}/send`**: sends the digest to the host's phone. It needs an
   `ADMIN_TOKEN` header and answers 404 without it.
5. **Warm-up**: the forecaster is fitted when the server starts.
6. Text in English and Kiswahili. The Kiswahili is mine and needs a native speaker's check.
   Day and month names stay in English for now.

### **6.3.3 Predictions (written before the run)**
1. `pytest -q` reports 10 passed (8 plus 2 new).
2. Bibi's USSD main menu has five options, the fifth in Kiswahili ("5. Utabiri wangu").
3. Option 5 from Bibi returns a Kiswahili outlook of at most 182 characters (including "END "),
   containing "25 Dec" and "4,700".
4. Option 5 from Tasha (a traveler) and from an unknown number returns the English
   "List your place first (option 2)."
5. The digest preview is in Kiswahili, at most 306 characters, and contains "Utulivu" (the quiet
   stretch) and "4,700".
6. `POST .../send` without the token returns 404; with it, 200 with `sent: true`, the
   destination +254700000001 and language "sw", and the message appears in the server terminal
   (console mode) or goes through the sandbox.
7. The server starts without printing a "[warm-up]" error.

### **6.3.4 Decision rule**
If predictions 1 to 6 hold, accept and start the dashboard. If not, fix first. Prediction 7
is a check, not a gate.

In [2]:
import httpx
from pathlib import Path

BASE = "http://127.0.0.1:8020"
CODE = "XESD2C"
BIBI, TASHA, NOBODY = "+254700000001", "+254711111111", "+254799999999"

env = {}
for line in Path("../.env").read_text().splitlines():
    if "=" in line and not line.lstrip().startswith("#"):
        key, value = line.split("=", 1)
        env[key.strip()] = value.strip()
TOKEN = env.get("ADMIN_TOKEN", "")


def ussd(phone, text=""):
    r = httpx.post(f"{BASE}/ussd", data={
        "sessionId": "nb-6-3", "serviceCode": "*384*1#", "phoneNumber": phone, "text": text,
    }, timeout=60)
    return r.text


print("Bibi, main menu:\n" + ussd(BIBI) + "\n")
out = ussd(BIBI, "5")
print(f"Bibi, option 5 ({len(out)} characters):\n{out}\n")
print("Tasha (traveler), option 5:\n" + ussd(TASHA, "5") + "\n")
print("Unknown number, option 5:\n" + ussd(NOBODY, "5") + "\n")

d = httpx.get(f"{BASE}/advice/{CODE}/digest", timeout=60).json()
print(f"SMS digest preview ({d['language']}, {d['length']} characters):\n{d['text']}\n")

no_token = httpx.post(f"{BASE}/advice/{CODE}/send", timeout=60)
print("send without token:", no_token.status_code)
with_token = httpx.post(f"{BASE}/advice/{CODE}/send", headers={"x-admin-token": TOKEN}, timeout=60)
print("send with token:", with_token.status_code,
      {k: v for k, v in with_token.json().items() if k != "text"})

Bibi, main menu:
CON Karibu Loca-T
1. Tafuta makazi
2. Orodhesha mahali pangu
3. Nafasi zangu
4. Lugha/Language
5. Utabiri wangu

Bibi, option 5 (130 characters):
END Bibi Akinyi Home
Shughuli nyingi karibuni: Fri 27 Nov ~69%, Sat 28 Nov ~69%.
Kilele: Fri 25 Dec ~97% imejaa. Jaribu KES 4,700.

Tasha (traveler), option 5:
END List your place first (option 2).

Unknown number, option 5:
END List your place first (option 2).

SMS digest preview (sw, 216 characters):
Loca-T, Bibi Akinyi Homestay: shughuli nyingi karibuni Fri 27 Nov ~69%, Sat 28 Nov ~69%. Kilele Fri 25 Dec ~97% imejaa. Jaribu KES 4,700. Utulivu 4 Jan hadi 24 Jan (usiku 15): toa zawadi ya rufaa. (data ya majaribio)

send without token: 404
send with token: 200 {'sent': True, 'to': '+254700000001', 'language': 'sw', 'length': 216}
